In [ ]:
"""
Visualization 2
This script performs province-level Mantel tests and correlation analysis between yield components and environmental/socioeconomic features.
Compared with Book1, which analyzes the entire study region, this script repeats the analysis for each province separately.
The workflow includes:
    - Load province-specific yield and feature datasets
    - Perform Mantel tests between yield components and features
    - Compute Pearson correlation matrix of features
    - Construct a combined network-correlation heatmap
    - Export datasets and final visualization
"""

### Load required libraries.
library(linkET)
suppressPackageStartupMessages(library(tidyverse))

### Load province-level datasets.
### Mantel test between yield components and feature set.
province <- c("hebei")
for (i in province) {
    yield <- read_csv(paste0("./book/s2/", i, "_yield_dataset.csv"), show_col_types = FALSE)
    feature <- read_csv(paste0("./book/s2/", i, "_feat_dataset.csv"), show_col_types = FALSE)

    mantel <- mantel_test(yield, feature,
        spec_dist = "euclidean",
        env_dist = "euclidean",
        spec_select = list("YO" = 1, "TY" = 2, "DY" = 3),
        permutations = 999
    ) %>%
        mutate(
            rd = cut(abs(r),
                breaks = c(-Inf, 0.2, Inf),
                labels = c("<= 0.2", "> 0.2")
            ),
            pd = cut(p,
                breaks = c(-Inf, 0.005, 0.01, 0.05, Inf),
                labels = c("<= 0.005", "0.005 - 0.01", "0.01 - 0.05", "> 0.05")
            )
        )
    write_csv(mantel, paste0("./book/s2/", i, "_mantel_result.csv"))
}


In [ ]:
### Load additional visualization libraries.
library(linkET)
library(ggnewscale)
library(RColorBrewer)
library(viridis)
suppressPackageStartupMessages(library(tidyverse))

### Generate correlation network heatmaps.
province <- c("hebei")
for (i in province) {
    feat_path <- paste0("./book/s2/", i, "_feat_dataset.csv")
    mantel_path <- paste0("./book/s2/", i, "_mantel_result.csv")
    cor_path <- paste0("./book/s2/", i, "_correl_matrix.csv")
    plot_path <- paste0("./book/s2/", i, "_net_heat_plot.svg")

    feature <- read_csv(feat_path, show_col_types = FALSE)
    mantel <- read_csv(mantel_path, show_col_types = FALSE) %>%
        mutate(pd = factor(pd, levels = c("<= 0.005", "0.005 - 0.01", "0.01 - 0.05", "> 0.05")))

    cor_mat <- correlate(feature)
    cor_df <- cor_mat %>%
        unclass() %>%
        as.data.frame() %>%
        tibble::rownames_to_column(var = "Var")
    write_csv(cor_df, cor_path)

    p <- qcorrplot(cor_mat, type = "upper", diag = FALSE, grid_col = NA) +
        geom_point(shape = 21, size = 4, fill = NA, stroke = 0.5, color = "black") +
        geom_point(aes(size = abs(r), fill = r),
            shape = 21, stroke = 0.4, color = "black"
        ) +
        scale_size(range = c(1, 3), guide = "none") +
        new_scale("size") +
        geom_couple(
            data = mantel,
            aes(color = pd, size = rd),
            label.size = 2.46,
            label.family = "Arail",
            label.fontface = 2,
            nudge_x = 0.5,
            curvature = nice_curvature(by = "from")
        ) +
        scale_fill_gradientn(
            limits = c(-0.8, 0.8),
            breaks = seq(-0.8, 0.8, 0.4),
            colors = rev(brewer.pal(11, "RdBu"))
        ) +
        scale_size_manual(values = c(0.3, 1)) +
        scale_color_manual(values = viridis(8, alpha = 0.88)) +
        guides(
            fill = guide_colorbar(
                title = "Pearson's r",
                title.vjust = 3,
                keyheight = unit(1.8, "cm"),
                keywidth = unit(0.3, "cm"),
                order = 1
            ),
            size = guide_legend(
                title = "Mantel's r",
                order = 2,
                keyheight = unit(0.3, "cm")
            ),
            colour = guide_legend(
                title = "Mantel's p",
                order = 3,
                keyheight = unit(0.3, "cm")
            )
        ) +
        theme(
            legend.box.spacing = unit(3, "pt"),
            axis.text = element_text(size = 6),
            legend.title = element_text(size = 6),
            legend.text = element_text(size = 5)
        )
    ggsave(p, file = plot_path, width = 5, height = 4)
}
